In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#Loading Csv File
df = pd.read_csv('C:/Users/USER/Desktop/Kaggle/New Data/datasets/historical_data.csv')
df.isnull().sum()

market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526
dtype: int64

In [3]:
#Making a seoarate list for filling missing values in numerical columns
col=df.select_dtypes(include=['float64','int64']).columns.tolist()
print(col)

['market_id', 'store_id', 'order_protocol', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders', 'estimated_order_place_duration', 'estimated_store_to_consumer_driving_duration']


In [4]:
#Filling up missing values in numerical columns
df[col] = df[col].fillna(df[col].median())
df[col].isnull().sum()

market_id                                       0
store_id                                        0
order_protocol                                  0
total_items                                     0
subtotal                                        0
num_distinct_items                              0
min_item_price                                  0
max_item_price                                  0
total_onshift_dashers                           0
total_busy_dashers                              0
total_outstanding_orders                        0
estimated_order_place_duration                  0
estimated_store_to_consumer_driving_duration    0
dtype: int64

In [5]:
#Backward filling actual_delivery_time
df['actual_delivery_time'] = df['actual_delivery_time'].bfill()
df['actual_delivery_time'].isnull().sum()

np.int64(0)

In [6]:
#Creating total_estimated_time for reducing features
df['total_estimated_time'] = df['estimated_order_place_duration']+df['estimated_store_to_consumer_driving_duration']

In [7]:
#Checking number of columns
df.columns

Index(['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'total_estimated_time'],
      dtype='object')

In [8]:
#Creating a list for categorical columns to fill up missing values 
col_2=df.select_dtypes(include=['object']).columns.tolist()
print(col_2)

['created_at', 'actual_delivery_time', 'store_primary_category']


In [9]:
#Changing time features to datetime data type
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

In [10]:
#Creating a new feature by taking difference of to features and extracting seconds
df['delivery_time'] = (df['actual_delivery_time']-df['created_at']).dt.total_seconds()
df['delivery_time']

0         3779.0
1         4024.0
2         1781.0
3         3075.0
4         2390.0
           ...  
197423    3907.0
197424    3383.0
197425    3008.0
197426    3907.0
197427    2228.0
Name: delivery_time, Length: 197428, dtype: float64

In [59]:
#Calculation of Upper and Lower limit for total_items
Q2 = df['delivery_time'].quantile(0.25)
Q4 = df['delivery_time'].quantile(0.75)
IQR24 = Q4-Q2
L2 = Q2-(1.5*IQR24)
H4 = Q4+(1.5*IQR24)
print(L2,H4)

220.375 4965.375


In [61]:
df = df[df['delivery_time']<H4]
df['delivery_time'] = df['delivery_time'].clip(lower=0)
df['delivery_time'].describe()

count    132697.000000
mean       2588.511330
std         824.218598
min           0.000000
25%        1981.000000
50%        2478.000000
75%        3097.000000
max        4965.000000
Name: delivery_time, dtype: float64

In [11]:
print(col_2)

['created_at', 'actual_delivery_time', 'store_primary_category']


In [12]:
#Backward Filling for categorical columns
df[col_2] = df[col_2].bfill()
df.isnull().sum()

market_id                                       0
created_at                                      0
actual_delivery_time                            0
store_id                                        0
store_primary_category                          0
order_protocol                                  0
total_items                                     0
subtotal                                        0
num_distinct_items                              0
min_item_price                                  0
max_item_price                                  0
total_onshift_dashers                           0
total_busy_dashers                              0
total_outstanding_orders                        0
estimated_order_place_duration                  0
estimated_store_to_consumer_driving_duration    0
total_estimated_time                            0
delivery_time                                   0
dtype: int64

Since some columns has outliers, we also need to remove them to improve
model performance.

In [13]:
#Calculation of Upper and Lower limit for total_items
Q1 = df['total_items'].quantile(0.25)
Q3 = df['total_items'].quantile(0.75)
IQR = Q3-Q1
L = Q1-(1.5*IQR)
H = Q3+(1.5*IQR)
print(L,H)

-1.0 7.0


In [14]:
#Instead of removing directly , we are using filtering method to change total_items as per need.
df = df[df['total_items']<H]
df['total_items'].describe()

count    182817.000000
mean          2.686736
std           1.400527
min           1.000000
25%           2.000000
50%           2.000000
75%           4.000000
max           6.000000
Name: total_items, dtype: float64

In [15]:
#Calculation of Upper and Lower Limit for subtotal
Q11 = df['subtotal'].quantile(0.25)
Q31 = df['subtotal'].quantile(0.75)
IQR1 = Q31-Q11
L1 = Q11-(1.5*IQR1)
H1 = Q31+(1.5*IQR1)
print(L1,H1)

-1287.5 5812.5


In [16]:
#Filtering subtotal for improving model performance.
df=df[df['subtotal']<H1]
df['subtotal'].describe()

count    177037.000000
mean       2295.491140
std        1189.059907
min           0.000000
25%        1350.000000
50%        2051.000000
75%        3020.000000
max        5812.000000
Name: subtotal, dtype: float64

In [17]:
#Calculation of Upper and Lower Limit for min_item_price
Q12 = df['min_item_price'].quantile(0.25)
Q32 = df['min_item_price'].quantile(0.75)
IQR2 = Q32-Q12
L2 = Q12-(1.5*IQR2)
H2 = Q32+(1.5*IQR2)
print(L2,H2)

-662.5 1957.5


In [18]:
#Filtering min_item_price for improving model performance
df=df[df['min_item_price']<H2]

In [19]:
#We also need to remove negative values, because negative values won't allow to create further features
#and also can affect model performance.
df['min_item_price'] = df['min_item_price'].clip(lower=0)
df['min_item_price'].describe()

count    172925.000000
mean        663.914431
std         410.230523
min           0.000000
25%         305.000000
50%         600.000000
75%         942.000000
max        1956.000000
Name: min_item_price, dtype: float64

In [20]:
#Checking outliers for max_item_price
df['max_item_price'].describe()

count    172925.000000
mean       1098.309487
std         442.246980
min           0.000000
25%         799.000000
50%        1050.000000
75%        1310.000000
max        5500.000000
Name: max_item_price, dtype: float64

In [21]:
#Calculation of upper and lower limit for max_item_price
Q13 = df['max_item_price'].quantile(0.25)
Q33 = df['max_item_price'].quantile(0.75)
IQR3 = Q33-Q13
L3 = Q13-(1.5*IQR3)
H3 = Q33+(1.5*IQR3)
print(L3,H3)

32.5 2076.5


In [22]:
#Filtering max_item_price for improving model performance
df=df[df['max_item_price']<H3]
df['max_item_price'].describe()

count    168635.000000
mean       1061.379227
std         373.645592
min           0.000000
25%         799.000000
50%        1025.000000
75%        1299.000000
max        2076.000000
Name: max_item_price, dtype: float64

In [23]:
#Creating new list for further data cleaning
col_3 = df.select_dtypes(include=['float64','int64']).columns.tolist()
print(col_3)

['market_id', 'store_id', 'order_protocol', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders', 'estimated_order_place_duration', 'estimated_store_to_consumer_driving_duration', 'total_estimated_time', 'delivery_time']


In [24]:
#Checking outliers for total_onshift_dashers
df['total_onshift_dashers'].describe()

count    168635.000000
mean         43.405195
std          32.846413
min          -4.000000
25%          18.000000
50%          37.000000
75%          61.000000
max         171.000000
Name: total_onshift_dashers, dtype: float64

In [25]:
#Hence the feature has also negative values as well as outliers, we need to handle.
df['total_onshift_dashers']=df['total_onshift_dashers'].clip(lower=0)

In [26]:
#Creating upper and lower limit for total_onshift_dashers
Q14 = df['total_onshift_dashers'].quantile(0.25)
Q34 = df['total_onshift_dashers'].quantile(0.75)
IQR4 = Q34-Q14
L4 = Q14-(1.5*IQR4)
H4 = Q34+(1.5*IQR4)
print(L4,H4)

-46.5 125.5


In [27]:
#Filtering total_onshift_dashers 
df=df[df['total_onshift_dashers']<H4]
df['total_onshift_dashers'].describe()

count    164526.000000
mean         41.114505
std          29.812389
min           0.000000
25%          18.000000
50%          37.000000
75%          58.000000
max         125.000000
Name: total_onshift_dashers, dtype: float64

In [28]:
print(col_3)

['market_id', 'store_id', 'order_protocol', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders', 'estimated_order_place_duration', 'estimated_store_to_consumer_driving_duration', 'total_estimated_time', 'delivery_time']


In [29]:
#Checking outliers for total_busy_dashers
df['total_busy_dashers'].describe()

count    164526.000000
mean         38.704563
std          28.777294
min          -5.000000
25%          16.000000
50%          34.000000
75%          56.000000
max         154.000000
Name: total_busy_dashers, dtype: float64

In [30]:
#First we remove negative values , then we handle outliers in total_busy_dashers by filtering method
df['total_busy_dashers'] = df['total_busy_dashers'].clip(lower=0)
Q15 = df['total_busy_dashers'].quantile(0.25)
Q35 = df['total_busy_dashers'].quantile(0.75)
IQR5 = Q35-Q15
L5 = Q15-(1.5*IQR5)
H5 = Q35+(1.5*IQR5)
print(L5,H5)

-44.0 116.0


In [31]:
df=df[df['total_busy_dashers']<H5]
df['total_busy_dashers'].describe()

count    162604.000000
mean         37.706336
std          27.420509
min           0.000000
25%          16.000000
50%          34.000000
75%          55.000000
max         115.000000
Name: total_busy_dashers, dtype: float64

In [32]:
print(col_3)

['market_id', 'store_id', 'order_protocol', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders', 'estimated_order_place_duration', 'estimated_store_to_consumer_driving_duration', 'total_estimated_time', 'delivery_time']


In [33]:
#Checking outliers for total_outstanding_orders
df['total_outstanding_orders'].describe()

count    162604.000000
mean         50.860735
std          44.048718
min          -6.000000
25%          18.000000
50%          41.000000
75%          71.000000
max         243.000000
Name: total_outstanding_orders, dtype: float64

In [34]:
#Removing negative values and outliers from total_outstanding_orders
df['total_outstanding_orders']=df['total_outstanding_orders'].clip(lower=0)
Q16 = df['total_outstanding_orders'].quantile(0.25)
Q36 = df['total_outstanding_orders'].quantile(0.75)
IQR6 = Q36-Q16
L6 = Q16-(1.5*IQR6)
H6 = Q36+(1.5*IQR6)
print(L6,H6)

-61.5 150.5


In [35]:
df=df[df['total_outstanding_orders']<H6]
df['total_outstanding_orders'].describe()

count    155450.000000
mean         45.228099
std          35.964774
min           0.000000
25%          17.000000
50%          40.000000
75%          65.000000
max         150.000000
Name: total_outstanding_orders, dtype: float64

In [36]:
#Creating new time features and adjusting older ones to decrease mean_absolute_error
df['prep_time_estimate'] = df['total_estimated_time']-df['estimated_store_to_consumer_driving_duration']
df['order_load'] = df['total_outstanding_orders']/(df['total_onshift_dashers']+1)

In [37]:
#Since the feature has no null values , so we had to check for outliers and negative values
df['order_load'].describe()

count    155450.000000
mean          1.094427
std           0.426868
min           0.000000
25%           0.869565
50%           1.078947
75%           1.325000
max          23.500000
Name: order_load, dtype: float64

In [38]:
Q17 = df['order_load'].quantile(0.25)
Q37 = df['order_load'].quantile(0.75)
IQR7 = Q37-Q17
L7 = Q17-(1.5*IQR7)
H7 = Q37+(1.5*IQR7)
print(L7,H7)

0.18641304347826082 2.0081521739130435


In [39]:
df=df[df['order_load']<H7]
df['order_load'].describe()

count    153468.000000
mean          1.076744
std           0.375734
min           0.000000
25%           0.864865
50%           1.078947
75%           1.312500
max           2.000000
Name: order_load, dtype: float64

In [40]:
#Since the feature has no null values , so we had to check for outliers and negative values
df['prep_time_estimate'].describe()

count    153468.000000
mean        310.191414
std          90.673093
min           0.000000
25%         251.000000
50%         251.000000
75%         446.000000
max        2715.000000
Name: prep_time_estimate, dtype: float64

In [41]:
Q18 = df['prep_time_estimate'].quantile(0.25)
Q38 = df['prep_time_estimate'].quantile(0.75)
IQR8 = Q38-Q18
L8 = Q18-(1.5*IQR8)
H8 = Q38+(1.5*IQR8)
print(L8,H8)

-41.5 738.5


In [42]:
df=df[df['prep_time_estimate']<H8]
df['prep_time_estimate'].describe()

count    153456.000000
mean        310.104447
std          90.035294
min           0.000000
25%         251.000000
50%         251.000000
75%         446.000000
max         732.000000
Name: prep_time_estimate, dtype: float64

In [43]:
df.dtypes

market_id                                              float64
created_at                                      datetime64[ns]
actual_delivery_time                            datetime64[ns]
store_id                                                 int64
store_primary_category                                  object
order_protocol                                         float64
total_items                                              int64
subtotal                                                 int64
num_distinct_items                                       int64
min_item_price                                           int64
max_item_price                                           int64
total_onshift_dashers                                  float64
total_busy_dashers                                     float64
total_outstanding_orders                               float64
estimated_order_place_duration                           int64
estimated_store_to_consumer_driving_duration           

In [44]:
#We also create some additonal order feature , to increase the feature importance of time_features
df['avg_item_price'] = df['subtotal']/df['total_items']
df['order_complexity'] = df['total_items']*df['num_distinct_items']

In [45]:
#We can clearly see , that we had to handle outliers only
df['avg_item_price'].describe()

count    153456.000000
mean        918.634259
std         387.836098
min           0.000000
25%         646.000000
50%         875.000000
75%        1141.000000
max        5299.000000
Name: avg_item_price, dtype: float64

In [46]:
Q19 = df['avg_item_price'].quantile(0.25)
Q39 = df['avg_item_price'].quantile(0.75)
IQR9 = Q39-Q19
L9 = Q19-(1.5*IQR9)
H9 = Q39+(1.5*IQR9)
print(L9,H9)

-96.5 1883.5


In [47]:
df=df[df['avg_item_price']<H9]
df['avg_item_price'].describe()

count    150635.000000
mean        895.313456
std         348.309412
min           0.000000
25%         640.000000
50%         868.000000
75%        1120.000000
max        1883.333333
Name: avg_item_price, dtype: float64

In [48]:
#Same goes for this feature
df['order_complexity'].describe()

count    150635.000000
mean          7.719501
std           7.294128
min           1.000000
25%           2.000000
50%           4.000000
75%           9.000000
max          36.000000
Name: order_complexity, dtype: float64

In [49]:
Q110 = df['order_complexity'].quantile(0.25)
Q310 = df['order_complexity'].quantile(0.75)
IQR10 = Q310-Q110
L10 = Q110-(1.5*IQR10)
H10 = Q310+(1.5*IQR10)
print(L10,H10)

-8.5 19.5


In [50]:
df=df[df['order_complexity']<H10]
df['order_complexity'].describe()

count    137404.000000
mean          5.997249
std           4.707134
min           1.000000
25%           2.000000
50%           4.000000
75%           9.000000
max          18.000000
Name: order_complexity, dtype: float64

In [51]:
#Checking columns again, so that we can decide which features actually useless and we can remove
df.columns

Index(['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'total_estimated_time',
       'delivery_time', 'prep_time_estimate', 'order_load', 'avg_item_price',
       'order_complexity'],
      dtype='object')

In [52]:
#Removing useless features to imrpove model
df=df.drop(['created_at','actual_delivery_time','estimated_order_place_duration',
            'estimated_store_to_consumer_driving_duration',
            'total_outstanding_orders','min_item_price','max_item_price',
            'store_id'],axis=1)
df.columns

Index(['market_id', 'store_primary_category', 'order_protocol', 'total_items',
       'subtotal', 'num_distinct_items', 'total_onshift_dashers',
       'total_busy_dashers', 'total_estimated_time', 'delivery_time',
       'prep_time_estimate', 'order_load', 'avg_item_price',
       'order_complexity'],
      dtype='object')

From this notebook, we  cleaned the data, so that we can form further business insights and models for better use.
1.) Handling the missing values , by using df.isnull().sum() first to check the missing values, later, we used df[col]=df[col].fillna(df[col].median()) in numerical features, and df[col] = df[col].bfill() for categorical features.
2.) Handling outliers, by checking first df[col].describe() to check outliers in  numerical features , and then we find out IQR or Interquartile Range (Difference between 3rd and 1st quartile) following by Creation of Upper Limit(Q3+(1.5*IQR)) and Lower Limit(Q1-(1.5*IQR)) to further limit the range of the data , so that we can apply models like Linear Regression or other Regression models on this data with least error.
3.) For Data Encoding, we have Ordinal Encoder to handle the single categorical columns.
4.) Creating new features, so that the model will have high train accuracy as well as test accuracy(in Regression we dtermine this by checking errors not accuracy Score).